# Figure: soil mineral N versus N&#8322;O emissions

Accumulates N&#8322;O fluxes over the periods covered by the PRS probe burials and
the soil samplings, and relates them to NO&#8323;&#8315; supply and Nmin stocks.

**Input** `../data/raw/fluxes_meteo_mgmt.parquet`, `../data/raw/PRS_nutrients_agg.csv`,
`../data/raw/Nmin_agg.csv`
**Output** `../figures/soilN_vs_N2O.png`


# LOAD LIBRARIES

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
from sklearn.metrics import r2_score
from scipy.stats import linregress, t, probplot, shapiro
import matplotlib.dates as mdates
from diive.core.io.files import load_parquet
import string

# READ DATA

In [ ]:
# Define period of interest
start_date = '2024-01-01'
end_date = '2024-08-22'

## EDDY DATA

In [ ]:
filename = "../data/raw/fluxes_meteo_mgmt.parquet"
df = load_parquet(filename)

# Observed non-gapfilled N2O fluxes
df['FN2O_parcelA_obs'] = df['FN2O_L3.3_CUT_50_QCF0_parcelA']
df['FN2O_parcelB_obs'] = df['FN2O_L3.3_CUT_50_QCF0_parcelB']

# Define the columns we are interested
rename_dict = {
    'FN2O_L3.3_CUT_50_QCF_parcelA_gfXGBoost': 'FN2O_parcelA',
    'FN2O_L3.3_CUT_50_QCF_parcelB_gfXGBoost': 'FN2O_parcelB',
    'FN2O_parcelA_obs': 'FN2O_parcelA_obs',
    'FN2O_parcelB_obs': 'FN2O_parcelB_obs',
    'FN2O_L3.3_CUT_50_QCF0': 'FN2O_observed',
    'ts_0.15': 'TS',
    'swc_0.15': 'SWC',
    'prec': 'PREC'
}

df.rename(columns=rename_dict, inplace=True)

# Convert units
n2o_cols = [c for c in rename_dict.values() if c.startswith('FN2O')]
for col in n2o_cols:
    df[col] = df[col] * 60 * 30 * 28.014 * 1e-9 * 1e-3 * 1e4 # nmol N2O m-2 s-1 → kg N2O-N ha-1 30min-1

# gpp_cols = [c for c in rename_dict.values() if c.startswith('GPP')]
# for col in gpp_cols:
#     df[col] = df[col] * 60 * 30 * 12.011 * 1e-6 # umol CO2 m-2 s-1 → g C m-2 30min-1

# Select columns to keep
cols = [c for c in rename_dict.values()]

# Select period
df = df.loc[start_date:end_date, cols].copy()

# Plot
indat = df.copy()
start = '2024-02-01'
end = '2024-08-01'
indat = indat[start:end]
indat[n2o_cols].plot(x_compat=True, figsize=(10, 2*len(n2o_cols)), subplots=True, sharey=True, sharex=True);

df

## PRS

### AGGREGATED

In [ ]:
prs = pd.read_csv("../data/raw/PRS_nutrients_agg.csv")

prs["burial_dt"] = pd.to_datetime(prs["burial_dt"], errors='coerce')
prs["retrieval_dt"] = pd.to_datetime(prs["retrieval_dt"], errors='coerce')
prs["date"] = pd.to_datetime(prs["date"], errors='coerce')
prs['NO3_PRS'] = prs['no3-n_mean']
prs['NO3_PRS_se'] = prs['no3-n_se']
prs['NO3_PRS_count'] = prs['no3-n_count']

# Select period
end = '2024-08-22' # end  when the grass was sowed
mask = (prs['burial_dt'] > start_date) & (prs['retrieval_dt'] < end)
prs = prs[mask].copy()

# Add index for annotation: same index for same dates across parcels
unique_dates = prs['date'].sort_index().unique()
date_index_map = {date: idx for idx, date in enumerate(unique_dates, 1)}
prs['sampling_index'] = prs['date'].map(date_index_map)

prs

## Nmin

In [ ]:
nmin = pd.read_csv('../data/raw/Nmin_agg.csv')

# Fix date to a datetime object
nmin['date'] = pd.to_datetime(nmin['date'], format='%Y-%m-%d')
# set timestamp as index
nmin.set_index('date', inplace=True)
# Select columns to keep
unit = 'kgN_ha' # 'kgN_ha' or 'mgN_kg'
cols_to_keep = [c for c in nmin.columns if ('Nmin' in c or 'no3' in c or 'nh4' in c) and unit in c] + ['parcel', 'depth']
nmin = nmin[cols_to_keep].copy()
# Select period
nmin = nmin[start_date:end_date].copy()
# Plot
nmin.plot(x_compat=True, figsize=(10, 4));
nmin

# PLOTTING CONFIGS

In [ ]:
from plot_config import COLORS, EVENT_LINES, LABELS, add_event_lines

# Labels for y axis
y_labels = {
    'Nmin_kgN_ha': 'SMN at 0-15 cm\n(kg N ha$^{-1}$)',
    'no3-n': 'Available NO$_3$-N\n(μg N 10 cm$^{-2}$)',
}

# ANALYSIS

## Nmin vs. FN2O-EDDY

In [ ]:
# Compute the aggregate (sum or average) in a window of days at the time of the sampling
n_days = 7

cols_mean = ['TS', 'SWC']   # only these should be averaged

# 1) Daily aggregation: sum for all, mean for TS & SWC
df_daily = (
    df[cols]
    .resample('D')
    .agg(lambda s: s.mean() if s.name in cols_mean else s.sum())
)

# 2) Rolling window: same logic (sum vs mean)
df_agg = (
    df_daily
    .rolling(window=n_days, center=True)
    .agg(lambda s: s.mean() if s.name in cols_mean else s.sum())
)
df_nmin = nmin.join(df_agg, how='outer')

df_nmin

In [ ]:
# Define which Nmin columns to use
nmin_cols = [f'no3_{unit}', f'nh4_{unit}', f'Nmin_{unit}']

# Loop over each depth
for depth in df_nmin['depth'].dropna().unique():
    # Copy the dataset for this depth
    indat = df_nmin[df_nmin['depth'] == depth].copy()
    
    # Create parcel-specific FN2O column (A or B)
    indat['FN2O'] = np.where(
        indat['parcel'] == 'A',
        indat['FN2O_parcelA'],
        indat['FN2O_parcelB']
    )

    # Number of variables to plot
    num_vars = len(nmin_cols)

    # Create a shared y-axis figure
    fig, axes = plt.subplots(nrows=1, ncols=num_vars, figsize=(5 * num_vars, 5), sharey=True)

    # If only one column, axes is not a list
    if num_vars == 1:
        axes = [axes]

    # Loop through each column in `nmin_cols`
    for i, c in enumerate(nmin_cols):
        ax = axes[i]  # Select subplot

        # Drop rows with missing values in relevant columns
        tmp = indat[[f'{c}_mean', f'{c}_se', f'{c}_count', 'FN2O', 'parcel']].dropna()
        if tmp.empty:
            continue

        # Extract the relevant data
        X = tmp[f'{c}_mean'].values  # Independent variable (1D array)
        y = tmp['FN2O'].values  # Dependent variable
        se = tmp[f'{c}_se'].values  # x error
        n = tmp[f'{c}_count'].values # n for x

        # Calculate confidence interval for x
        # Calculate critical t-value for 95% CI (two-tailed, alpha=0.025)
        t_crit = np.where(n > 1, t.ppf(0.975, df=n-1), 1.96)
        X_err = se * t_crit

        # If you want to use +- 1 se
        X_err = se

        # Colors based on parcel
        colors = tmp['parcel'].map(COLORS).values

        # Perform linear regression
        slope, intercept, r_value, p_value, std_err = linregress(X, y)
        y_pred = slope * X + intercept  # Predicted values

        # Calculate R² value
        r2 = r2_score(y, y_pred)

        # Scatter plot
        ax.scatter(X, y, c=colors)

        # X error bars
        ax.errorbar(
            X, y, xerr=X_err,
            fmt='none', ecolor='lightgray', elinewidth=1, capsize=2, zorder=1
        )

        # Regression line
        ax.plot(X, y_pred, color='red',
                label=f'R² = {r2:.2f}, p = {p_value:.3f}')

        # Labels
        ax.set_xlabel(c)
        if i == 0:
            ax.set_ylabel('N$_{2}$O-N flux (kg N week$^{-1}$ ha$^{-1}$)')

        # Title
        ax.set_title(f'Depth: {depth} cm')

        ax.legend()

    plt.tight_layout()
    plt.show()

In [ ]:
# Prepare the data
col = 'Nmin'
indat = df_nmin[df_nmin['depth']=='0-15'].dropna(subset=[f'{col}_{unit}_mean', 'FN2O_parcelA', 'FN2O_parcelB']).copy()
indat['FN2O'] = np.where(
    indat['parcel'] == 'A',
    indat['FN2O_parcelA'],
    indat['FN2O_parcelB']
)
X = indat[f'{col}_{unit}_mean'].values
y = indat['FN2O'].values

# Fit the model and calculate residuals
slope, intercept, r_value, p_value, std_err = linregress(X, y)
y_pred = slope * X + intercept
residuals = y - y_pred

# Set up diagnostic plots
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Plot A: Residuals vs Fitted (Checks Linearity and Homoscedasticity)
axes[0].scatter(y_pred, residuals, alpha=0.7, edgecolor='k')
axes[0].axhline(0, color='red', linestyle='--')
axes[0].set_xlabel('Fitted Values (Predicted cum_FN2O)')
axes[0].set_ylabel('Residuals')
axes[0].set_title('Residuals vs Fitted')

# Plot B: Histogram of Residuals (Checks Normality)
axes[1].hist(residuals, bins='auto', edgecolor='black', alpha=0.7)
axes[1].set_xlabel('Residuals')
axes[1].set_ylabel('Frequency')
axes[1].set_title('Histogram of Residuals')

# Plot C: Q-Q Plot (Checks Normality)
probplot(residuals, dist="norm", plot=axes[2])
axes[2].set_title('Normal Q-Q Plot')

plt.tight_layout()
plt.show()

# Formal statistical test for Normality (Shapiro-Wilk)
stat, p_sw = shapiro(residuals)
print(f"Shapiro-Wilk Test for Normality: W-Statistic={stat:.4f}, p-value={p_sw:.4f}")

if p_sw > 0.05:
    print(" -> Assumption met: Residuals appear normally distributed (fail to reject H0).")
else:
    print(" -> Assumption violated: Residuals do NOT appear normally distributed (reject H0).")

In [ ]:
def plot_NMIN_vs_FN2O(ax, df, col, depth):

    indat = df.copy()
    indat = indat[indat['depth'] == depth].sort_values(f'{col}_{unit}_mean').copy()
    indat = indat.dropna(subset=[f'{col}_{unit}_mean', f'{col}_{unit}_std', f'{col}_{unit}_count',
                                 'FN2O_parcelA', 'FN2O_parcelB', 'parcel'])
    
    # Add index for annotation: same index for same dates across parcels
    unique_dates = indat.sort_index().index.unique()
    date_index_map = {date: idx for idx, date in enumerate(unique_dates)}
    indat['sampling_index'] = indat.index.map(date_index_map)

    # Parcel-specific logic: use parcel A/B specific FN2O
    indat['FN2O'] = np.where(
        indat['parcel'] == 'A',
        indat['FN2O_parcelA'],
        indat['FN2O_parcelB']
    )

    # Extract data
    X = indat[f'{col}_{unit}_mean'].values
    y = indat['FN2O'].values

    # Colors based on parcel
    colors = indat['parcel'].map(COLORS).values

    # Linear regression
    slope, intercept, r_value, p_value, std_err = linregress(X, y)
    y_pred = slope * X + intercept
    r2 = r2_score(y, y_pred)

    # Plot (fixed marker size)
    ax.scatter(X, y, s=60, color=colors, alpha=0.5, zorder=2)
    ax.plot(X, y_pred, color='black', linestyle='--')

    # Annotate each point with its sampling index in the form of a letter
    for xi, yi, idx in zip(X, y, indat['sampling_index']):
        ax.text(xi+0.1, yi, string.ascii_uppercase[idx], ha='left', va='bottom', fontsize=12)

    # R² and p-value
    ax.annotate(f'$R^2$ = {r2:.2f}\np = {p_value:.4f}', xy=(0.02, 0.9),
                xycoords='axes fraction', ha='left', va='top')

    # Labels
    ax.set_ylabel('N$_{2}$O-N flux (kg N week$^{-1}$ ha$^{-1}$)')
    ax.set_xlabel('SMN at 0-15 cm (kg N ha $^{-1}$)')
    ax.tick_params(labelsize=11)

In [ ]:
# Nmin 0-15 in wheat
fig, ax = plt.subplots(figsize=(8, 6))
plot_NMIN_vs_FN2O(ax, df_nmin, 'Nmin', '0-15')
plt.tight_layout()
plt.show()

## N-PRS vs. FN2O-EDDY

In [ ]:
# map parcel -> column name in df
col_map = {"A": "FN2O_parcelA", "B": "FN2O_parcelB"}

def sum_fn2o_for_window(row):
    col = col_map[row["parcel"]]
    start, end = row["burial_dt"], row["retrieval_dt"]
    # sum FN2O in [start, end]; change to start:end if you want end-exclusive
    return df.loc[start:end, col].sum(skipna=True)

# build output
prs_eddy = prs.copy()
end = '2024-08-01' # need to remove the last period because it was during the period when the station was off --> no measured N2O data + different duration (one week more)
mask = (prs_eddy['burial_dt'] > start_date) & (prs_eddy['retrieval_dt'] < end)
prs_eddy = prs_eddy[mask].copy()
prs_eddy["FN2O_cum"] = prs_eddy.apply(sum_fn2o_for_window, axis=1)

# optional: nicer column order / names
prs_eddy = prs_eddy[["parcel", "burial_dt", "retrieval_dt", "date", "NO3_PRS", "FN2O_cum", "sampling_index"]].rename(
    columns={"FN2O_cum": "cum_FN2O"}
)

# remove NaN
prs_eddy.dropna(inplace=True)

prs_eddy


In [ ]:
# Prepare the data
indat = prs_eddy.dropna(subset=['NO3_PRS', 'cum_FN2O']).copy()
X = indat['NO3_PRS'].values
y = indat['cum_FN2O'].values

# Fit the model and calculate residuals
slope, intercept, r_value, p_value, std_err = linregress(X, y)
y_pred = slope * X + intercept
residuals = y - y_pred

# Set up diagnostic plots
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Plot A: Residuals vs Fitted (Checks Linearity and Homoscedasticity)
axes[0].scatter(y_pred, residuals, alpha=0.7, edgecolor='k')
axes[0].axhline(0, color='red', linestyle='--')
axes[0].set_xlabel('Fitted Values (Predicted cum_FN2O)')
axes[0].set_ylabel('Residuals')
axes[0].set_title('Residuals vs Fitted')

# Plot B: Histogram of Residuals (Checks Normality)
axes[1].hist(residuals, bins='auto', edgecolor='black', alpha=0.7)
axes[1].set_xlabel('Residuals')
axes[1].set_ylabel('Frequency')
axes[1].set_title('Histogram of Residuals')

# Plot C: Q-Q Plot (Checks Normality)
probplot(residuals, dist="norm", plot=axes[2])
axes[2].set_title('Normal Q-Q Plot')

plt.tight_layout()
plt.show()

# Formal statistical test for Normality (Shapiro-Wilk)
stat, p_sw = shapiro(residuals)
print(f"Shapiro-Wilk Test for Normality: W-Statistic={stat:.4f}, p-value={p_sw:.4f}")

if p_sw > 0.05:
    print(" -> Assumption met: Residuals appear normally distributed (fail to reject H0).")
else:
    print(" -> Assumption violated: Residuals do NOT appear normally distributed (reject H0).")

In [ ]:
def plot_PRS_vs_FN2O(ax, prs_eddy):
    """Plot NO3_PRS vs cum_FN2O with single regression for both parcels."""
    
    # Remove NaN values
    indat = prs_eddy.dropna(subset=['NO3_PRS', 'cum_FN2O']).copy()
    
    # Single regression across all data
    X = indat['NO3_PRS'].values
    y = indat['cum_FN2O'].values
    
    slope, intercept, r_value, p_value, std_err = linregress(X, y)
    r2 = r_value**2
    
    # Line for plotting
    x_line = np.linspace(X.min(), X.max(), 100)
    y_line = slope * x_line + intercept
    ax.plot(x_line, y_line, color='black', linestyle='--')
    
    # Plot both parcels with different colors
    for p in ['A', 'B']:
        m = indat['parcel'] == p
        X_p = indat.loc[m, 'NO3_PRS'].values
        y_p = indat.loc[m, 'cum_FN2O'].values
        
        # Scatter
        ax.scatter(
            X_p, y_p,
            s=60, alpha=0.5, linewidth=0.6,
            color=COLORS[p], zorder=2
        )
    
    # Annotate each point with its index
    for xi, yi, idx in zip(indat['NO3_PRS'].values, indat['cum_FN2O'].values, indat['sampling_index'].values):
        ax.text(xi+0.5, yi, str(idx), ha='left', va='bottom', fontsize=12)
    
    ax.set_xlabel('Available NO$_3$-N (μg N 10 cm$^{-2}$)')
    ax.set_ylabel('N$_2$O-N flux (kg N ha$^{-1}$)')
    ax.annotate(f'$R^2$ = {r2:.2f}\np = {p_value:.4f}', xy=(0.02, 0.9),
                xycoords='axes fraction', ha='left', va='top')

In [ ]:
# Create figure and axis
fig, ax = plt.subplots(figsize=(8, 6))

# Call the function with axis and dataframe
plot_PRS_vs_FN2O(ax, prs_eddy)

# Display the plot
plt.tight_layout()
plt.show()

# FINAL FIG

## top panel

In [ ]:
def plot_prs_timeseries(ax, df, col, bar_width=3):
    """
    Plots PRS data as side-by-side bars for Parcel A and B (same style as
    plot_nmin_timeseries). Burial duration is shown as a shaded background
    band + a small day-count label, decoupled from bar height so BAU/VRA
    values stay easy to compare at a constant bar width.
    bar_width defines the width of ONE bar in days.
    """
    df = df.copy()
    df['mean_dt'] = df['burial_dt'] + (df['retrieval_dt'] - df['burial_dt']) / 2
    mcol = f"{col}_mean"
    ecol = f"{col}_se"

    # Unique burial windows (one per sampling_index, shared by both parcels)
    windows = (df.dropna(subset=['sampling_index'])
                 .drop_duplicates(subset=['sampling_index'])
                 .sort_values('sampling_index'))
    windows['duration_days'] = (windows['retrieval_dt'] - windows['burial_dt']).dt.days

    # Show the burial windows as vertical lines and soft shaded background (to show duration + any gaps, e.g. around harvest)
    for _, w in windows.iterrows():
        ax.axvline(w['burial_dt'], color='0.6', lw=0.6, zorder=0)
        ax.axvline(w['retrieval_dt'], color='0.6', lw=0.6, zorder=0)
        ax.axvspan(w['burial_dt'], w['retrieval_dt'], color='0.6', alpha=0.1, zorder=0)

    # Define the time shift for each parcel to place them side-by-side
    offsets = {
        "A": -pd.Timedelta(days=bar_width / 2),
        "B": pd.Timedelta(days=bar_width / 2)
    }

    # Plot the time-series for each parcel
    for parcel in ("A", "B"):
        sub = df[df['parcel'] == parcel].dropna(subset=[mcol, ecol])
        if sub.empty: continue

        sub = sub.sort_values('mean_dt')
        color = COLORS.get(parcel, "black")

        # Apply the time offset to the dates
        shifted_dates = sub['mean_dt'] + offsets[parcel]

        # Plot bars and error bars
        ax.bar(
            shifted_dates,
            sub[mcol],
            yerr=sub[ecol],
            width=bar_width,
            align='center',
            color=color,
            alpha=0.5,
            edgecolor='black',
            linewidth=0.8,
            capsize=3,
            zorder=1,
            error_kw={'elinewidth': 2, 'ecolor': color, 'zorder': 2},
            label=LABELS[parcel]
        )

    # Add the sampling index (top) + burial duration in days (right below it)
    for _, w in windows.iterrows():
        ax.text(
            w['mean_dt'], 0.9, str(int(w['sampling_index'])),
            transform=ax.get_xaxis_transform(),
            ha='center', va='bottom', color='black',
            clip_on=False
        )

    # Format the Y-axis label
    ax.set_ylabel(y_labels[col])

    return ax

# Plot
fig, ax = plt.subplots(figsize=(14, 4))
plot_prs_timeseries(ax, prs, 'no3-n')
ax.xaxis.set_major_locator(mdates.MonthLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b'))
plt.tight_layout()
plt.show()

In [ ]:
def plot_nmin_timeseries(ax, df, col, bar_width=3):
    """
    Plots instantaneous Nmin data as side-by-side bars for Parcel A and B.
    bar_width defines the width of ONE bar in days.
    """
    df = df.copy()
    mcol = f"{col}_mean"
    ecol = f"{col}_se"
    
    # Define the time shift for each parcel to place them side-by-side
    offsets = {
        "A": -pd.Timedelta(days=bar_width / 2),
        "B": pd.Timedelta(days=bar_width / 2)
    }
    
    # Plot the time-series for each parcel
    for parcel in ("A", "B"):
        sub = df[df['parcel'] == parcel].dropna(subset=[mcol, ecol])
        if sub.empty: continue
            
        sub = sub.sort_index()
        color = COLORS.get(parcel, "black")
        
        # Apply the time offset to the dates
        shifted_dates = sub.index + offsets[parcel]
        
        # Plot bars and error bars
        ax.bar(
            shifted_dates, 
            sub[mcol], 
            yerr=sub[ecol], 
            width=bar_width,      
            align='center',       
            color=color, 
            alpha=0.5,            
            edgecolor='black',
            linewidth=0.8,
            capsize=3,
            zorder=1,
            error_kw={'elinewidth': 2, 'ecolor': color, 'zorder': 2},
            label=LABELS[parcel] 
        )

    # Add the sequential letters perfectly centered on the TRUE sampling date
    unique_dates = df.dropna(subset=[mcol]).index.unique().sort_values()
    letters = string.ascii_uppercase
    
    for i, dt in enumerate(unique_dates):
        letter = letters[i % 26] 
        ax.text(
            dt, 0.9, letter,
            transform=ax.get_xaxis_transform(),
            ha='center', va='bottom', color='black',
            clip_on=False 
        )

    # Format the Y-axis label 
    ax.set_ylabel(y_labels[col])

    # Add legend to the top right, above the plot, side-by-side
    ax.legend(
        loc='lower right', 
        bbox_to_anchor=(1.0, 1.4), 
        frameon=False
    )
    
    return ax
# Plot
fig, ax = plt.subplots(figsize=(14, 4))
plot_nmin_timeseries(ax, nmin, 'Nmin_kgN_ha')
ax.xaxis.set_major_locator(mdates.MonthLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b'))
plt.tight_layout()
plt.show()

## combined fig

In [ ]:
plt.rcParams.update({'font.size': 14})  # Set global font size
fig = plt.figure(figsize=(14, 14))
gs = GridSpec(nrows=4, ncols=2, figure=fig,
              height_ratios=[3, 3, 0.5, 6],  # nearly zero for spacing row
              hspace=0,
              wspace=0.05)

# Top two rows: timeline plots (stacked)
ax_nmin_timeline = fig.add_subplot(gs[0, :])
ax_prs_timeline = fig.add_subplot(gs[1, :], sharex=ax_nmin_timeline)

# Spacer row (invisible): row 2 [2,:] — do nothing here

# Bottom row: result plots (side by side)
ax_NMIN = fig.add_subplot(gs[3, 0])
ax_PRS = fig.add_subplot(gs[3, 1])

# Plot timeseries data
plot_prs_timeseries(ax_prs_timeline, prs, 'no3-n')
ax_prs_timeline.set_ylim(0, 550)  # Adjust y-axis limit for better visualization
plot_nmin_timeseries(ax_nmin_timeline, nmin, 'Nmin_kgN_ha')
ax_nmin_timeline.set_ylim(0, 25)  # Adjust y-axis limit for better visualization

# Add management
start_ts = pd.to_datetime('2024-03-01')
end_ts = pd.to_datetime('2024-08-21')
add_event_lines(ax_nmin_timeline, start_ts=start_ts, end_ts=end_ts, mgmt=True, height=1.02, x_spacing_days=4)
add_event_lines(ax_prs_timeline, start_ts=start_ts, end_ts=end_ts, mgmt=False, height=1.02, x_spacing_days=4)

# Format bottom timeline axis
ax_nmin_timeline.tick_params(axis='x', which='both', bottom=False, labelbottom=False)
ax_prs_timeline.xaxis.set_major_locator(mdates.MonthLocator())
ax_prs_timeline.xaxis.set_major_formatter(mdates.DateFormatter('%b'))
secax = ax_prs_timeline.secondary_xaxis('bottom')
secax.xaxis.set_major_locator(mdates.YearLocator())
secax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
secax.tick_params(axis='x', length=0, pad=30)

# Plot bottom panels
plot_PRS_vs_FN2O(ax_PRS, prs_eddy)
plot_NMIN_vs_FN2O(ax_NMIN, df_nmin, 'Nmin', '0-15')

# Move y-axis of PRS plot to the right
ax_PRS.yaxis.tick_right()
ax_PRS.yaxis.set_label_position("right")

# Labels
x = 0.005
y = 0.98
ax_nmin_timeline.text(x, y, '(a)', transform=ax_nmin_timeline.transAxes, fontweight='bold', va='top', ha='left')
ax_prs_timeline.text(x, y, '(b)', transform=ax_prs_timeline.transAxes, fontweight='bold', va='top', ha='left')
ax_NMIN.text(x, y, '(c)', transform=ax_NMIN.transAxes, fontweight='bold', va='top', ha='left')
ax_PRS.text(x, y, '(d)', transform=ax_PRS.transAxes, fontweight='bold', va='top', ha='left')

# Align axis labels
fig.align_ylabels([ax_prs_timeline, ax_nmin_timeline, ax_PRS])
fig.align_xlabels([ax_NMIN, ax_PRS])

# Show or save
plt.savefig("../figures/soilN_vs_N2O.png", dpi=300, bbox_inches='tight')
plt.show()
